# Teacher's Notebook — Prosthetics Data Engineering Workshop

**Who this is for:** A facilitator delivering this workshop live for the first time. Read each markdown cell before running the code below it — the text tells you what to say and why the code is structured the way it is.

**How to use it:** Run top-to-bottom. Every talking point is written as prose you can read aloud or paraphrase. Every function is built in its own small cell so you can pause, explain, and ask questions before moving on.

**What participants will build:** By the end of the session, participants have a working ETL pipeline that ingests messy patient data, cleans it, masks names with SHA-256, and joins it to a device inventory table.

---
## Step 0 — Setup

**Say:** "Before we start, make sure you can see the file `prosthetics_data.csv` in your folder. It is already there — you don't need to run anything to generate it. Let's load it and see what we're dealing with."

Why `pathlib.Path` over a hardcoded string: the solution notebook is one folder deeper than the participant workbook, so the CSV lives at `../prosthetics_data.csv` relative to the solution but at `prosthetics_data.csv` relative to the participant. Using a candidate list handles both without changing any code.

In [ ]:
from pathlib import Path
import re
import hashlib
import pandas as pd

In [ ]:
# Find the CSV regardless of which folder this notebook is opened from
csv_candidates = [
    Path('prosthetics_data.csv'),
    Path('..') / 'prosthetics_data.csv',
]
for candidate in csv_candidates:
    if candidate.exists():
        csv_path = candidate
        break
else:
    raise FileNotFoundError('Could not find prosthetics_data.csv in this folder or the parent.')

df = pd.read_csv(csv_path)
df.head()

**Say:** "Before writing a single line of cleaning code, we always inspect. Three calls tell us almost everything."

- `df.info()` — column names, dtypes, and non-null counts. Messy types show up immediately (object where you'd expect int or datetime).
- `df.isna().sum()` — counts missing values per column. Two columns have NaN: `Full_Name` (PT-007) and `Age_Input` (PT-010).
- `df.head()` — just enough rows to see the variety of formats without overwhelming the screen.

**Point out live:**
- `Age_Input` is dtype `object` (string) even though most values look like numbers. That's a red flag.
- `Implant_Date` also `object` — dates were stored as free-text, so we'll find six or seven different formats.
- There is an `Amputation_Pos` column (amputation position codes: AK, BK, TT, etc.) that we will not clean today — point it out so participants know it's there but tell them it's out of scope.

In [ ]:
df.info()
print('\nMissing values per column:')
print(df.isna().sum())

---
## Section 1 — Ingestion & Pipelines (ETL vs ELT)

**Say:** "Before we touch the data, let's establish *why* the pipeline design matters in healthcare."

**ETL — Extract, Transform, Load:**
- Data is validated and privacy-masked *before* it reaches the destination.
- Right choice when the destination is a shared cloud system and you can't guarantee raw data never lands there.
- Cost: you need to know your transformations upfront.

**ELT — Extract, Load, Transform:**
- Raw data lands first; transformations happen later inside a warehouse.
- Right choice when data volumes are large and storage is cheap — transform on demand, not upfront.
- Risk in healthcare: raw patient records (names, diagnoses) sit in the warehouse until the transform runs.

**Ask the room:** "If this charity sends patient records to a cloud analytics platform, should they ETL or ELT — and why?"

Expected answer: ETL, because we want names and other PII masked *before* data reaches the cloud. The workshop will follow that path: clean first, mask names, *then* join and analyse.

**No code cell here** — this is conceptual framing. Move on once there are a few answers from the room.

---
## Section 2 — Data Cleaning

**Say:** "Now the messiest part. We have three cleaning problems:
1. `Age_Input` — integers, birth years, and freeform text all in one column.
2. `Implant_Date` — six or seven different date formats including named months with ordinal suffixes.
3. `Amputation_Cause` — inconsistent casing and trailing spaces.

We will write one small function per problem. That makes each piece independently testable."

### 2a — Parse `Age_Input`

**Show participants the raw values first:**

In [ ]:
# What does the column actually contain?
df[['Patient_ID', 'Age_Input']]

**Say:** "We see four kinds of value:
- A plain number that is already an age: `28`, `34`, `19`, ...
- A birth year: `1995`, `1988`, `2001` — these need converting to age.
- Free text with a number inside: `'around 50'`, `'maybe 27'`, `'approx 36'`, `"Don't know, maybe 40"`.
- `NaN` (PT-010 is genuinely missing).

The rule is:
1. If the value is NaN → return None.
2. If it's a pure number and > 1900 → it's a birth year; subtract from 2026.
3. If it's a pure number and ≤ 1900 → it's already an age.
4. Otherwise, pull the first run of digits out of the text.

**Ask:** 'What threshold should we use to tell an age from a birth year?' — 1900 is a safe lower bound; no living prosthetics patient was born before 1900, and no realistic age reaches 1900."

In [ ]:
CURRENT_YEAR = 2026

def parse_age_input(value):
    """Convert the messy Age_Input column to a clean integer age."""
    if pd.isna(value):
        return None

    # Numeric value — already an int/float, or a string like '34'
    if isinstance(value, (int, float)) and not pd.isna(value):
        numeric_value = int(value)
    else:
        match = re.search(r'\d+', str(value))
        if not match:
            return None
        numeric_value = int(match.group())

    if numeric_value < 100:
        return numeric_value           # already an age
    if numeric_value > 1900:
        return CURRENT_YEAR - numeric_value   # birth year
    return numeric_value               # edge case: 100–1900, return as-is

**Say:** "Let's test it on a few representative inputs before we apply it to the whole column. This catches bugs early and makes the logic visible."

In [ ]:
# Quick sanity checks before applying to the whole column
test_cases = [
    (28,                    28),   # plain integer age
    ('34',                  34),   # age as string
    (1995,                  31),   # birth year
    ("Don't know, maybe 40", 40),  # free text
    ('around 50',           50),
    (float('nan'),         None),  # missing
]
for value, expected in test_cases:
    result = parse_age_input(value)
    status = 'OK' if result == expected else f'FAIL (expected {expected})'
    print(f'{str(value):30} -> {str(result):6}  {status}')

In [ ]:
df['Age_Years'] = df['Age_Input'].apply(parse_age_input)
df[['Patient_ID', 'Age_Input', 'Age_Years']]

### 2b — Parse `Implant_Date`

**Say:** "Dates are the classic real-world pain. Show the raw column and count the formats."

In [ ]:
# How many different formats do we actually have?
df['Implant_Date'].unique()

**Say:** "You might be tempted to pass everything to `pd.to_datetime(dayfirst=True)` and call it done. There is a subtle bug: `dayfirst=True` with `format='mixed'` also reorders the *month and day components* inside year-first strings. So `2023-09-01` becomes January 9th, not September 1st. Demo this live:"

In [ ]:
# DEMONSTRATION OF THE PITFALL — do not use this in the actual pipeline
demo = pd.Series(['2023-09-01', '2023/09/01'])
print('With dayfirst=True (WRONG for year-first dates):')
print(pd.to_datetime(demo, format='mixed', dayfirst=True).tolist())
print()
print('With dayfirst=False (correct for year-first, wrong for DD/MM/YYYY):')
print(pd.to_datetime(demo, format='mixed', dayfirst=False).tolist())

**Say:** "There is no single `pd.to_datetime` call that handles *both* `DD/MM/YYYY` and `YYYY/MM/DD` safely. The fix is to detect which category each value belongs to before handing it to pandas.

Two special steps:
1. Strip ordinal suffixes (`'5th'` → `'5'`, `'12th'` → `'12'`) so that `'April 5th 2024'` can be parsed.
2. Check if the value starts with a 4-digit year using a regex. If yes, reorder it to ISO (`YYYY-MM-DD`) directly, *before* calling `pd.to_datetime`. Then pandas never gets a chance to misinterpret it."

In [ ]:
# Step 1: strip ordinal suffixes
# 'May 12th 2024' -> 'May 12 2024', 'April 5th 2024' -> 'April 5 2024'
demo_dates = pd.Series(['May 12th 2024', 'April 5th 2024', '2024-06-15'])
demo_dates.str.replace(r'(\d+)(st|nd|rd|th)', r'\1', regex=True)

In [ ]:
# Step 2: the regex that detects year-first formats
# Matches: 2023-09-01, 2024/04/18, 2024-06-15 — anything starting with 4 digits
pattern = r'^(\d{4})[-/](\d{1,2})[-/](\d{1,2})$'
for v in ['2023-09-01', '2023/09/01', '2024/04/18', '12/05/2024', 'May 12 2024']:
    matched = bool(re.match(pattern, v.strip()))
    print(f'{v:20} year-first? {matched}')

In [ ]:
def parse_implant_date(raw):
    """Return YYYY-MM-DD string for any date format found in this dataset."""
    if pd.isna(raw):
        return None
    s = re.sub(r'(\d+)(st|nd|rd|th)', r'\1', str(raw).strip())
    # Year-first: handle before dayfirst=True to avoid month/day swap
    m = re.match(r'^(\d{4})[-/](\d{1,2})[-/](\d{1,2})$', s)
    if m:
        y, mo, d = m.groups()
        return f'{y}-{int(mo):02d}-{int(d):02d}'
    # Named months and DD/MM/YYYY — pandas can handle these with dayfirst=True
    try:
        return pd.to_datetime(s, dayfirst=True).strftime('%Y-%m-%d')
    except Exception:
        return None

In [ ]:
# Test against every format in the dataset
date_tests = [
    ('12/05/2024',   '2024-05-12'),
    ('2024-06-15',   '2024-06-15'),
    ('May 12th 2024','2024-05-12'),
    ('07/11/2023',   '2023-11-07'),
    ('2023/09/01',   '2023-09-01'),   # the tricky one
    ('March 3 2024', '2024-03-03'),
    ('20/02/2024',   '2024-02-20'),
    ('April 5th 2024','2024-04-05'),
    ('2024/04/18',   '2024-04-18'),
    ('15/07/2024',   '2024-07-15'),
]
for raw, expected in date_tests:
    got = parse_implant_date(raw)
    status = 'OK' if got == expected else f'FAIL (expected {expected})'
    print(f'{raw:22} -> {got}  {status}')

In [ ]:
df['Implant_Date_Clean'] = df['Implant_Date'].apply(parse_implant_date)
df[['Patient_ID', 'Implant_Date', 'Implant_Date_Clean']]

### 2c — Standardise `Amputation_Cause`

**Say:** "This one is the simplest of the three. The values are correct, just inconsistently cased and padded. A single chained call handles it.

**Ask:** 'What does `str.strip()` do?' — removes leading and trailing whitespace.

**Ask:** 'What does `str.lower()` do?' — normalises `'TRAUMA'`, `'Trauma'`, `' TRAUMA'` all to `'trauma'`.

**Point out:** PT-012 has `'diabets'` — a genuine typo. That kind of error requires a value-mapping step (a dictionary of corrections) which is beyond today's scope, but participants should notice it in the output."

In [ ]:
df['Amputation_Cause_Clean'] = df['Amputation_Cause'].str.lower().str.strip()
df[['Patient_ID', 'Amputation_Cause', 'Amputation_Cause_Clean']]

**Point out live:** Run `value_counts()` and show the `'diabets'` typo appearing alongside `'diabetes'`. This illustrates why a simple lowercase/strip is not always enough — you sometimes need a controlled vocabulary or a spell-checker step.

In [ ]:
df['Amputation_Cause_Clean'].value_counts()

---
## Section 3 — Cybersecurity & Cloud Privacy (PII Masking)

**Say:** "We have cleaned data but it still contains patient names — Personally Identifiable Information (PII). Before this data moves anywhere outside our secure environment, the names must go.

There are two approaches: deletion (remove the column entirely) and pseudonymisation (replace the value with a stable token derived from it). We use pseudonymisation so that if two datasets mention the same person, the tokens still match — without re-exposing the name.

SHA-256 is a cryptographic hash function. Given the same input it always produces the same 64-character hex string. It is one-way: you cannot reverse it to recover the original name."

In [ ]:
# DEMO: what SHA-256 looks like
import hashlib

sample = 'John Doe'
digest = hashlib.sha256(sample.strip().lower().encode('utf-8')).hexdigest()
print(f'Input : {sample!r}')
print(f'Hash  : {digest}')
print(f'Length: {len(digest)} characters')

**Ask:** "Why do we call `.strip().lower()` before hashing?"

Expected answer: `'John Doe'` and `' john doe '` would produce different hashes. If the same person appears with slightly different casing or spacing in two datasets, we need their tokens to match. Normalising first ensures that.

**Then ask:** "Why do we encode to UTF-8?"

Expected answer: SHA-256 operates on bytes, not strings. `.encode('utf-8')` converts the Python string to a byte sequence.

In [ ]:
def anonymize_name(value):
    """Replace a patient name with its SHA-256 hash (normalised first)."""
    normalized = str(value).strip().lower()
    return hashlib.sha256(normalized.encode('utf-8')).hexdigest()

In [ ]:
# Apply before the join — this is the ETL 'Transform' step protecting PII
df['Full_Name'] = df['Full_Name'].apply(anonymize_name)
df[['Patient_ID', 'Full_Name']].head()

**Point out:** PT-007 had `NaN` in `Full_Name`. After `str(value)` it becomes the string `'nan'`, which still hashes consistently. In a real system you might want to return a sentinel instead (e.g., `None`), but for this workshop it is fine — the important concept is the hashing, not the NaN edge case.

---
## Section 4 — Relational Storage & Joins

**Say:** "We now have clean, privacy-safe patient records. The next step is enriching them by joining to the device inventory — a separate reference table that tells us what each device model is, what it costs, and who made it.

This is a classic relational join:
- `Device_Model_ID` is the **foreign key** in the patient table.
- `Device_Model_ID` is the **primary key** in the inventory table.
- An **inner join** keeps only rows that match on both sides.

**Ask:** 'Why inner join rather than left join?' — an inner join drops patients with a device ID not in our inventory (which would indicate a data entry error). We want to surface those anomalies rather than silently produce NaN columns."


In [ ]:
# Build the reference table
device_inventory = {
    'M210': {'Model_Name': 'AeroStep Lite',   'Cost_USD': 950,  'Manufacturer': 'Stride MedTech'},
    'M302': {'Model_Name': 'FlexFoot Alpha',  'Cost_USD': 1200, 'Manufacturer': 'Global Limb Works'},
    'M404': {'Model_Name': 'CrestFit Pro',    'Cost_USD': 1850, 'Manufacturer': 'OrthoSphere'},
    'M505': {'Model_Name': 'AdaptMax Field',  'Cost_USD': 2100, 'Manufacturer': 'Humanity Mobility Systems'},
}

device_inventory_df = pd.DataFrame.from_dict(device_inventory, orient='index').reset_index()
device_inventory_df = device_inventory_df.rename(columns={'index': 'Device_Model_ID'})
device_inventory_df = device_inventory_df[['Device_Model_ID', 'Model_Name', 'Cost_USD', 'Manufacturer']]
device_inventory_df

**Say:** "A Python dict cannot be passed directly to `pd.merge`. We convert it to a DataFrame first using `DataFrame.from_dict(..., orient='index')` — `orient='index'` means each dict key becomes a row, not a column.

After `reset_index()` the key becomes a regular column. Then we rename it to match the patient table's `Device_Model_ID` column so the join has something to match on."

In [ ]:
patient_device_df = pd.merge(df, device_inventory_df, on='Device_Model_ID', how='inner')
patient_device_df[['Patient_ID', 'Device_Model_ID', 'Model_Name', 'Cost_USD', 'Manufacturer']]

In [ ]:
# Confirm no rows were dropped (all 15 patients have a valid device ID)
print(f'Rows before merge: {len(df)}')
print(f'Rows after merge : {len(patient_device_df)}')
assert len(patient_device_df) == 15, 'Row count mismatch — check for unknown Device_Model_IDs'

---
## Wrap-Up

**Say:** "Let's look at the final DataFrame. Names are gone — replaced with hashes. Dates are standardised. Ages are numeric. The device details are joined in. This is a complete ETL pipeline in roughly 50 lines of Python."

Show the full pipeline result, then ask these debrief questions:

1. **'What's the difference between anonymisation and pseudonymisation?'** — Anonymisation means the original cannot be recovered. Pseudonymisation means it *can* be recovered with the right key. SHA-256 without a salt is one-way, so it leans toward anonymisation — but if an attacker knows the likely names they can pre-hash them and compare.

2. **'Which step in our pipeline was ETL and which was ELT?'** — The whole thing was ETL: we transformed (cleaned + masked) before loading (the merged table would be the final destination).

3. **'What cleaning problem did we NOT solve today?'** — The `'diabets'` typo in `Amputation_Cause`. Fixing typos requires either a controlled vocabulary mapping or fuzzy matching — both are valid follow-on topics.

4. **'What would break if someone added a new device model `M606` to a patient row?'** — The inner join would silently drop that patient. A left join would surface it as NaN, making the gap visible. Neither is wrong — it depends on your quality gate.

In [ ]:
# Final enriched, privacy-safe dataset
output_cols = [
    'Patient_ID', 'Full_Name', 'Age_Years',
    'Implant_Date_Clean', 'Amputation_Cause_Clean',
    'Device_Model_ID', 'Model_Name', 'Cost_USD', 'Manufacturer',
]
patient_device_df[output_cols]